In [ ]:
import pandas as pd

In [ ]:
#data = pd.read_csv("data/icpoes/decaf_coffees.csv", skiprows=6)
data = pd.read_csv("data/icpoes/roasted_coffees_QC.csv", skiprows=6)
data

In [ ]:
data.Description = data.Label.apply(lambda x: ' '.join(x.split()[:-1]))

In [ ]:
data['State'] = data.Description.dropna().apply(lambda x: x.split('; ')[0])
data['Processing'] = data.Description.dropna().apply(lambda x: x.split('; ')[1])

In [ ]:
data.iloc[0]

In [ ]:
data.Flags.value_counts()

In [ ]:
caldata = data[~data.Concentration.isin(('Uncal','####'))].copy()
caldata.Concentration = caldata.Concentration.astype(float)

In [ ]:
caldata[caldata.Concentration > 0]

In [ ]:
eldata = caldata.groupby(['Description','Element Label'])['Concentration'].mean().unstack().T
#eldata = caldata.groupby(['Label','Element Label'])['Concentration'].mean().unstack().T
#eldata = caldata.groupby(['State','Processing','Element Label'])['Concentration'].mean().unstack().T
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(eldata)
    #print(eldata[data.Label.drop_duplicates()])

In [ ]:
elmeans = eldata[eldata.columns[-3:]].mean(axis=1)
print(
    '< 0.05:', (elmeans < 0.05).sum(),
    '\n> 0.5:', (elmeans > 0.5).sum(),
    '\n0.05 =< =< 0.5:', ((0.05 < elmeans) & (elmeans < 0.5)).sum(),
)

In [ ]:
eldata.T.reset_index().drop('Processing',axis=1).groupby('State').mean().T

In [ ]:
processing_means = eldata.T.reset_index().drop('State',axis=1).groupby('Processing').mean().T
processing_means

In [ ]:
processing_means.rank(axis=1).mean().sort_values()